# YieldSAT en Google Colab: preparacion e inspeccion remota

Este cuaderno prepara datos en la maquina de Colab, sin guardar el ZIP en tu PC.
Se detiene si no reconoce un entorno Colab con `/content`.

Alcance: obtener solo `Preprocessed.zip` del paquete compartido, extraer un pais e inspeccionar un bloque pequeno del NetCDF. **Todavia no entrena los cinco modelos ni completa el protocolo de investigacion.**

1. Abre este archivo en https://colab.research.google.com mediante Archivo > Subir cuaderno.
2. Conecta un entorno estandar de Colab. Para esta inspeccion no necesitas GPU.
3. Ejecuta las celdas en orden. Pega el enlace compartido cuando se solicite.
4. La primera preparacion transfiere varios GB al disco de Colab. El cuaderno comprueba espacio antes de escribir.

No contiene el enlace privado ni credenciales. No montes un entorno de ejecucion local. Colab puede borrar su disco temporal al finalizar la sesion.


In [ ]:
import sys
from pathlib import Path

def require_colab():
    try:
        import google.colab
    except ImportError:
        raise RuntimeError("Ejecuta este cuaderno en Google Colab; no en tu equipo local.") from None
    if not Path('/content').is_dir():
        raise RuntimeError("No se encontro el disco remoto /content. Se detuvo la preparacion.")

require_colab()
import subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       'httpx>=0.27,<1', 'xarray>=2024.1,<2027',
                       'h5netcdf>=1.3,<2', 'h5py>=3.10,<4'])
ROOT = Path('/content/yieldsat_work')
ROOT.mkdir(exist_ok=True)
print('Directorio remoto:', ROOT)


## Acceso y lectura parcial

El enlace se introduce de forma oculta y permanece solo en memoria. Las lecturas HTTP exigen respuesta parcial `206`; si el servidor intenta enviar todo el ZIP, se detienen.


In [ ]:
import io, json, re, zipfile, hashlib, shutil, os
from getpass import getpass
from urllib.parse import urlsplit, urlunsplit, parse_qsl, urlencode
import httpx

share_url = getpass('Pega el enlace compartido de Both.zip: ').strip()
parts = urlsplit(share_url)
if parts.scheme != 'https' or not (parts.hostname or '').endswith('.sharepoint.com'):
    raise ValueError('Se esperaba un enlace HTTPS de SharePoint.')
query = dict(parse_qsl(parts.query))
query['download'] = '1'
download_url = urlunsplit((parts.scheme, parts.netloc, parts.path, urlencode(query), ''))
client = httpx.Client(follow_redirects=True, timeout=120)

class HTTPRangeFile(io.RawIOBase):
    def __init__(self, client, url, size):
        self.client, self.url, self.size = client, url, size
        self.position = 0
        self.transferred = 0
        self.budget = 64 * 1024**2
    def readable(self): return True
    def seekable(self): return True
    def tell(self): return self.position
    def seek(self, offset, whence=0):
        if whence not in (0, 1, 2): raise ValueError('Invalid seek')
        new = offset if whence == 0 else self.position + offset if whence == 1 else self.size + offset
        if new < 0: raise ValueError('Negative seek')
        self.position = new
        return new
    def read(self, count=-1):
        count = self.size-self.position if count < 0 else min(count, self.size-self.position)
        if count <= 0: return b''
        if count > 16*1024**2 or self.transferred+count > self.budget:
            raise RuntimeError('Lectura fuera del presupuesto de transferencia.')
        start, end = self.position, self.position+count-1
        try:
            with self.client.stream('GET', self.url, headers={
                'Range': f'bytes={start}-{end}', 'Accept-Encoding': 'identity'}) as response:
                expected = f'bytes {start}-{end}/{self.size}'
                if response.status_code != 206 or response.headers.get('content-range') != expected:
                    raise RuntimeError('SharePoint no entrego el rango solicitado; no se descargo el archivo completo.')
                chunks, received = [], 0
                for chunk in response.iter_bytes(65536):
                    received += len(chunk)
                    if received > count: raise RuntimeError('Respuesta mayor que el rango solicitado.')
                    chunks.append(chunk)
                data = b''.join(chunks)
        except httpx.HTTPError:
            raise RuntimeError('Fallo la conexion con SharePoint; vuelve a conectar el enlace.') from None
        if len(data) != count: raise RuntimeError('Respuesta parcial incompleta.')
        self.position += count
        self.transferred += count
        return data

try:
    with client.stream('GET', download_url, headers={
        'Range': 'bytes=-22', 'Accept-Encoding': 'identity'}) as response:
        match = re.fullmatch(r'bytes (\d+)-(\d+)/(\d+)', response.headers.get('content-range', ''))
        if response.status_code != 206 or not match:
            raise RuntimeError('El enlace no permite lectura parcial o requiere otra autorizacion.')
        total_bytes = int(match.group(3))
        resolved_url = str(response.url)
        tail = b''
        for chunk in response.iter_bytes(1024):
            tail += chunk
            if len(tail) > 65557: raise RuntimeError('Respuesta demasiado grande.')
except httpx.HTTPError:
    raise RuntimeError('No fue posible abrir el enlace compartido.') from None
remote = HTTPRangeFile(client, resolved_url, total_bytes)
outer_zip = zipfile.ZipFile(remote)
for item in outer_zip.infolist():
    print(item.filename, '| ZIP interno:', round(item.file_size/1e9, 3), 'GB')
print('Paquete remoto:', round(total_bytes/1e9, 3), 'GB')
print('Bytes leidos para el indice:', remote.transferred+len(tail))


## Preparar el ZIP preprocesado en el disco de Colab

Solo se transfiere el miembro `Preprocessed.zip`. No se descarga `Raw.zip` ni el paquete completo. El archivo parcial se elimina si falla la operacion. La lectura completa de cada miembro verifica su CRC y se guarda un recibo SHA-256 sin el enlace privado.


In [ ]:
def copy_zip_member(archive, info, destination):
    require_colab()
    destination = Path(destination).resolve()
    if ROOT.resolve() not in destination.parents:
        raise RuntimeError('El destino debe estar dentro del directorio remoto de trabajo.')
    receipt_path = destination.with_suffix(destination.suffix+'.receipt.json')
    if destination.exists():
        if receipt_path.exists():
            receipt = json.loads(receipt_path.read_text())
            if (destination.stat().st_size == info.file_size
                and receipt.get('crc32') == info.CRC and receipt.get('member') == info.filename):
                print('Archivo preparado en una ejecucion anterior:', destination.name)
                return destination
        raise RuntimeError('Ya existe un archivo sin recibo valido; usa otra carpeta de trabajo.')
    destination.parent.mkdir(parents=True, exist_ok=True)
    partial = destination.with_suffix(destination.suffix+'.part')
    if partial.exists(): partial.unlink()
    available = shutil.disk_usage(ROOT).free
    required = info.file_size + 1024**3
    if available < required:
        raise RuntimeError(f'Espacio insuficiente en Colab: libres {available/1e9:.1f} GB, requeridos {required/1e9:.1f} GB.')
    digest, written, last_progress = hashlib.sha256(), 0, 0
    try:
        with archive.open(info) as source, partial.open('xb') as output:
            while True:
                block = source.read(4*1024**2)
                if not block: break
                written += len(block)
                if written > info.file_size: raise RuntimeError('Tamano descomprimido inesperado.')
                output.write(block)
                digest.update(block)
                if written-last_progress >= 512*1024**2:
                    print(f'{destination.name}: {written/1e9:.2f} / {info.file_size/1e9:.2f} GB', flush=True)
                    last_progress = written
        if written != info.file_size: raise RuntimeError('Archivo incompleto.')
        os.replace(partial, destination)
    except Exception:
        if partial.exists(): partial.unlink()
        raise
    receipt_path.write_text(json.dumps({'member': info.filename, 'bytes': written,
                                       'crc32': info.CRC, 'sha256': digest.hexdigest()}, indent=2))
    return destination

preprocessed_info = outer_zip.getinfo('Preprocessed.zip')
remote.budget = remote.transferred + preprocessed_info.compress_size + 8*1024**2
preprocessed_path = copy_zip_member(outer_zip, preprocessed_info, ROOT/'Preprocessed.zip')
outer_zip.close()
client.close()


## Elegir un pais y comprobar su tamano real

Alemania se usa inicialmente para comprobar el formato, como en el tutorial oficial. Para estudiar maiz se debe elegir un pais que contenga ese cultivo, por ejemplo Argentina o Brasil. Cambia `COUNTRY` antes de extraer. El codigo no carga todos los paises.


In [ ]:
COUNTRY = 'Germany'
with zipfile.ZipFile(preprocessed_path) as archive:
    netcdfs = [entry for entry in archive.infolist() if entry.filename.lower().endswith('.nc')]
    for entry in netcdfs:
        print(entry.filename, '| descomprimido:', round(entry.file_size/1e9, 2), 'GB')
    chosen = [entry for entry in netcdfs if entry.filename.split('/')[0].casefold() == COUNTRY.casefold()]
    if len(chosen) != 1:
        raise RuntimeError('Selecciona un pais con un unico NetCDF en el inventario anterior.')
    netcdf_path = copy_zip_member(archive, chosen[0], ROOT/(COUNTRY+'.nc'))


## Inspeccion diferida y muestra limitada

Solo se materializa un bloque pequeno, no el cubo completo. Esta muestra sirve para comprobar el esquema, no para entrenar ni medir calidad predictiva. Antes del experimento deben elegirse parcelas completas, verificar unidades y fechas, fijar el horizonte de prediccion y separar entrenamiento/prueba sin mezclar pixeles de la misma parcela.


In [ ]:
import xarray as xr
import numpy as np

with xr.open_dataset(netcdf_path, engine='h5netcdf', cache=False) as dataset:
    if not {'sample', 'target'}.issubset(dataset.data_vars):
        raise RuntimeError('El archivo no coincide con el esquema ML-ready esperado.')
    dimensions = {name: int(size) for name, size in dataset.sizes.items()}
    bands = dataset['band'].values.astype(str).tolist() if 'band' in dataset.coords and dataset.sizes['band'] <= 1024 else []
    selection = {dim: slice(0, min(size, 16 if dim == 'index' else 2 if dim == 'time_step' else 8))
                 for dim, size in dataset['sample'].sizes.items()}
    probe = dataset['sample'].isel(**selection)
    if probe.nbytes > 1024**2: raise RuntimeError('La muestra excede 1 MB.')
    values = probe.values
    target_probe = dataset['target'].isel(index=slice(0, min(16, dimensions.get('index', 0)))).values
    report = {
        'country': COUNTRY, 'dimensions': dimensions, 'variables': list(dataset.data_vars),
        'bands': bands, 'target_attributes': {str(k): str(v) for k, v in dataset['target'].attrs.items()},
        'probe_shape': list(values.shape), 'probe_bytes': int(values.nbytes),
        'probe_missing_fraction': float(np.isnan(values).mean()),
        'target_probe_finite': bool(np.isfinite(target_probe).all()),
        'status': 'schema inspection only; no model trained; no full data quality audit',
    }
report_path = ROOT/'inspection.json'
report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding='utf-8')
print(json.dumps(report, indent=2, ensure_ascii=False))
print('Informe pequeno:', report_path)


La preparacion termina aqui. Comparte el contenido de `inspection.json` para conectar el esquema real al pipeline del proyecto. Los datos grandes permanecen en Colab.

Para guardar solo el informe pequeno en tu PC puedes ejecutar la siguiente celda. No descarga el NetCDF ni los ZIP.


In [ ]:
from google.colab import files
files.download(str(report_path))


## Auditoria ampliada
Esta celda revisa todos los pasos temporales en ventanas acotadas y recorre solo las etiquetas y grupos del NetCDF completo. Produce `inspection_quality.json`, sin repetir la descarga. No constituye entrenamiento ni una revision completa de todas las imagenes.


In [ ]:
"""Paste this file into one new cell of the existing YieldSAT Colab notebook.
No ZIP is downloaded. Only an audit JSON is exported to the browser.
Generated from backend/app/ml/yieldsat.py; do not edit helpers independently.
"""
from __future__ import annotations
import json
import re
from pathlib import Path
from typing import Any
import numpy as np

def _xarray():
    try:
        import xarray as xr
    except ImportError as exc:
        raise RuntimeError("YieldSAT NetCDF support requires backend/yieldsat-requirements.txt") from exc
    return xr

def _string(value: Any) -> str:
    if isinstance(value, np.generic):
        value = value.item()
    if isinstance(value, bytes):
        value = value.decode("utf-8", errors="replace")
    if value is None or (isinstance(value, float) and not np.isfinite(value)):
        return ""
    return str(value).strip()

def _code_map(array: Any, dataset_attrs: dict | None = None, variable_name: str | None = None) -> dict[str, str]:
    sources = [array.attrs]
    if dataset_attrs and variable_name:
        sources.append({key: dataset_attrs.get(f"{variable_name}_{key}")
                        for key in ("code_to_name", "mapping", "categories", "classes")})
        sources.append({"mapping": dataset_attrs.get(variable_name)})
    for attrs in sources:
        for key in ("code_to_name", "mapping", "categories", "classes"):
            value = attrs.get(key)
            if isinstance(value, str):
                try:
                    value = json.loads(value)
                except json.JSONDecodeError:
                    continue
            if isinstance(value, dict):
                return {_string(code): _string(name) for code, name in value.items()}
        # The official release stores categories directly as numeric attribute
        # names (e.g. crop.attrs['0'] = 'wheat'), rather than a nested mapping.
        direct = {_string(code): _string(name) for code, name in attrs.items()
                  if re.fullmatch(r"-?\d+", _string(code)) and _string(name)}
        if direct:
            return direct
    return {}

def _label(value: Any, mapping: dict[str, str]) -> str:
    raw = _string(value)
    return mapping.get(raw, raw)

def audit_yieldsat_quality(source: str | Path, *, probe_rows: int = 256,
                          metadata_batch_size: int = 8192,
                          max_probe_bytes: int = 32 * 1024 * 1024) -> dict[str, Any]:
    """Scan 1D labels/groups and probe all time steps in bounded index windows.

    The feature probe is a diagnostic sample, not a complete quality audit or
    a representative experimental subset. No features or large files are saved.
    """
    from collections import Counter

    path = Path(source).expanduser()
    if not path.is_file():
        raise FileNotFoundError("NetCDF not found; reuse the existing Colab runtime or prepare it again")
    for name, value in (("probe_rows", probe_rows), ("metadata_batch_size", metadata_batch_size),
                        ("max_probe_bytes", max_probe_bytes)):
        if type(value) is not int or value < 1:
            raise ValueError(f"{name} must be a positive integer")
    if metadata_batch_size > 65536:
        raise ValueError("metadata_batch_size must not exceed 65536")
    xr = _xarray()
    with xr.open_dataset(path, chunks=None, cache=False) as dataset:
        required = ("sample", "target", "band", "crop", "year", "country",
                    "farm_identifier", "field_shared_name")
        missing = [name for name in required if name not in dataset]
        if missing:
            raise ValueError(f"Missing variables: {', '.join(missing)}")
        sample = dataset["sample"]
        if set(sample.dims) != {"index", "time_step", "band"}:
            raise ValueError("sample must have dimensions index,time_step,band")
        if dataset["target"].dims != ("index",):
            raise ValueError("target must have dimension index")
        n, steps, channels = (int(dataset.sizes[key]) for key in ("index", "time_step", "band"))
        if min(n, steps, channels) < 1:
            raise ValueError("Dataset dimensions must be nonempty")
        rows = min(probe_rows, n)
        # Reserve both the float64 probe and an additional working array.
        estimated = rows * steps * channels * 16
        if estimated > max_probe_bytes:
            raise ValueError(f"Probe working arrays need about {estimated} bytes; reduce probe_rows")
        categorical_names = ("crop", "year", "country", "farm_identifier", "field_shared_name")
        mappings = {name: _code_map(dataset[name], dataset.attrs, name) for name in categorical_names}
        counts = {name: Counter() for name in ("crop", "year", "country")}
        farms, groups = set(), Counter()
        finite_targets = invalid_targets = negative_targets = 0
        target_min, target_max = float("inf"), float("-inf")
        for first in range(0, n, metadata_batch_size):
            end = min(n, first + metadata_batch_size)
            block = {}
            for name in categorical_names:
                var = dataset[name]
                if var.dims not in (("index",), ()):
                    raise ValueError(f"{name} must be scalar or indexed by index")
                values = [var.values.item()] * (end-first) if var.dims == () else var.isel(index=slice(first, end)).values
                block[name] = [_label(value, mappings[name]) for value in values]
                if any(not label for label in block[name]):
                    raise ValueError(f"{name} contains missing group labels")
            for name in counts:
                counts[name].update(block[name])
            farms.update(zip(block["country"], block["farm_identifier"]))
            groups.update(zip(block["country"], block["farm_identifier"], block["field_shared_name"],
                              block["crop"], block["year"]))
            if len(groups) > 20000 or len(farms) > 20000:
                raise ValueError("More than 20000 metadata groups; a different audit strategy is needed")
            values = np.asarray(dataset["target"].isel(index=slice(first, end)).values, dtype=float)
            valid = np.isfinite(values)
            finite_targets += int(valid.sum())
            invalid_targets += int((~valid).sum())
            negative_targets += int((values[valid] < 0).sum())
            if valid.any():
                target_min = min(target_min, float(values[valid].min()))
                target_max = max(target_max, float(values[valid].max()))

        windows = min(8, rows)
        width = max(1, rows // windows)
        starts = np.unique(np.linspace(0, max(0, n-width), windows, dtype=int))
        ranges, probed = [], 0
        finite_counts = np.zeros((steps, channels), dtype=np.int64)
        band_min = np.full(channels, np.inf)
        band_max = np.full(channels, -np.inf)
        band_names = [_string(value) for value in dataset["band"].values]
        optical = [i for i, name in enumerate(band_names) if re.fullmatch(r"B(?:\d{2}|8A)", name)]
        optical_counts = np.zeros(steps, dtype=np.int64)
        optical_rows_with_any = 0
        for start in starts:
            first = int(start)
            end = min(n, first + width, first + rows-probed)
            if end <= first:
                continue
            values = np.asarray(sample.isel(index=slice(first, end)).transpose(
                "index", "time_step", "band").values, dtype=float)
            finite = np.isfinite(values)
            finite_counts += finite.sum(axis=0)
            band_min = np.minimum(band_min, np.where(finite, values, np.inf).min(axis=(0, 1)))
            band_max = np.maximum(band_max, np.where(finite, values, -np.inf).max(axis=(0, 1)))
            if optical:
                optical_available = finite[:, :, optical].all(axis=2)
                optical_counts += optical_available.sum(axis=0)
                optical_rows_with_any += int(optical_available.any(axis=1).sum())
            ranges.append({"start": first, "stop": end})
            probed += end-first

        attrs_names = ("target", "sample", "times", "seeding_date", "harvesting_date")
        attributes = {name: {str(key): _string(value) for key, value in dataset[name].attrs.items()}
                      for name in attrs_names if name in dataset}
        if "times" in dataset:
            attributes["times"]["decoded_dtype"] = str(dataset["times"].dtype)
            for key in ("units", "calendar"):
                if key in dataset["times"].encoding:
                    attributes["times"]["encoding_" + key] = str(dataset["times"].encoding[key])
        units = _string(dataset["target"].attrs.get("units")) or None
        warnings = ["Feature coverage is based on bounded diagnostic windows, not the complete cube",
                    "Full-season inputs require a prediction cutoff before any forecast experiment",
                    "Rows are pixels; groups are field records and are not automatically distinct physical parcels"]
        if not units:
            warnings.append("Target unit is not declared in this file; document and supply an explicit unit before conversion")
        unknown_codes = [name for name in categorical_names
                         if np.issubdtype(dataset[name].dtype, np.number) and not mappings[name]]
        if unknown_codes:
            warnings.append("Numeric category mappings missing for: " + ", ".join(unknown_codes))
        if optical and optical_rows_with_any == 0:
            warnings.append("No complete optical observation found in the diagnostic windows")
        return {
            "audit_version": 1, "source_file": path.name, "source_file_bytes": path.stat().st_size,
            "dimensions": {"index": n, "time_step": steps, "band": channels},
            "metadata_rows_scanned": n, "crop_pixel_counts": dict(counts["crop"]),
            "year_pixel_counts": dict(counts["year"]), "country_pixel_counts": dict(counts["country"]),
            "farm_groups": len(farms), "field_record_groups": len(groups),
            "fields": [{"country": key[0], "farm": key[1], "field": key[2],
                        "crop": key[3], "year": key[4], "pixels": count}
                       for key, count in sorted(groups.items())],
            "categorical_mappings": mappings, "attributes": attributes,
            "target": {"declared_unit": units, "finite_count": finite_targets,
                       "invalid_count": invalid_targets, "negative_count": negative_targets,
                       "min": target_min if finite_targets else None,
                       "max": target_max if finite_targets else None},
            "probe": {"rows": probed, "ranges": ranges, "all_time_steps": True,
                      "all_bands": True, "working_arrays_upper_estimate_bytes": estimated,
                      "finite_fraction_by_time_step": (finite_counts.sum(axis=1)/(probed*channels)).tolist(),
                      "optical_band_names": [band_names[i] for i in optical],
                      "complete_optical_fraction_by_time_step": (optical_counts/probed).tolist() if optical else None,
                      "rows_with_any_complete_optical_observation": optical_rows_with_any if optical else None,
                      "bands": [{"name": name, "finite_fraction": float(finite_counts[:, i].sum()/(probed*steps)),
                                 "min": float(band_min[i]) if np.isfinite(band_min[i]) else None,
                                 "max": float(band_max[i]) if np.isfinite(band_max[i]) else None}
                                for i, name in enumerate(band_names)]},
            "warnings": warnings, "status": "diagnostic audit only; no training, unit conversion or feature scaling performed",
        }

# Run only inside the existing managed Colab runtime.
try:
    from google.colab import files
except ImportError:
    raise RuntimeError("Ejecuta esta celda en Colab, no en tu equipo local.") from None
if not Path('/content').is_dir():
    raise RuntimeError("No se encontro el disco remoto de Colab.")
source = Path(globals().get('netcdf_path', '/content/yieldsat_work/Germany.nc')).resolve()
if Path('/content') not in source.parents:
    raise RuntimeError("El NetCDF debe estar en el disco remoto /content.")
if not source.is_file():
    raise RuntimeError("No esta el NetCDF: usa la misma sesion de Colab que lo extrajo. Si termino, deberas prepararlo nuevamente en Colab.")
print('Auditando metadatos y 256 pixeles distribuidos; no se descarga ningun ZIP.')
quality_report = audit_yieldsat_quality(source, probe_rows=256)
quality_path = source.parent/'inspection_quality.json'
quality_path.write_text(json.dumps(quality_report, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
print(json.dumps({
    'crop_pixel_counts': quality_report['crop_pixel_counts'],
    'year_pixel_counts': quality_report['year_pixel_counts'],
    'field_record_groups': quality_report['field_record_groups'],
    'target': quality_report['target'],
    'finite_fraction_by_time_step': quality_report['probe']['finite_fraction_by_time_step'],
    'rows_with_any_complete_optical_observation': quality_report['probe']['rows_with_any_complete_optical_observation'],
    'warnings': quality_report['warnings'],
}, indent=2, ensure_ascii=False))
print('Se descarga solo el informe pequeno:', quality_path.name)
files.download(str(quality_path))
